In [ ]:
#Installation de gymnasium, l'environnement de simulation
!pip install swig
!pip install "gymnasium[classic-control]"
!pip install "gymnasium[box2d]"
!pip install "gymnasium[other]"
!mkdir -pv ~/.cache/xdgr
!export XDG_RUNTIME_DIR=$PATH:~/.cache/xdgr

In [ ]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.distributions import Categorical, Normal

import numpy as np


is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display
plt.ion()

# GPU ou CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

In [ ]:
def plot(data, data_mean, ylabel='Reward', show_result=False):
    plt.figure(1)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel(ylabel)
    plt.plot(np.array(data))
    plt.plot(np.array(data_mean))
    plt.pause(0.0001)
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [ ]:
# On crée l'environnement
continuous = False
env_name = "LunarLander-v3" 
if continuous:
    env = gym.make(env_name,continuous=True)
else:
    env = gym.make(env_name)

In [ ]:
class PolicyNetwork(nn.Module):

    
    def __init__(self, hidden_size, num_inputs, action_space):
        super(PolicyNetwork, self).__init__()
        self.action_space = action_space
        num_outputs = action_space
    
        self.linear1 = nn.Linear(num_inputs, hidden_size)
        self.linear2 = nn.Linear(hidden_size, num_outputs)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
    
        return F.softmax(self.linear2(x), dim=-1)


In [ ]:
class ValueNetwork(nn.Module):
    
    def __init__(self, hidden_size, num_inputs):
        super(ValueNetwork, self).__init__()
    
        self.linear1 = nn.Linear(num_inputs, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1) # Sortie scalaire V(s)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
        return self.linear2(x)

    def compute_ppo_loss(self, new_probs, old_probs, actions_onehot, advantages):
        prob = torch.sum(new_probs * actions_onehot, dim=1, keepdim=True)
        old_prob = torch.sum(old_probs * actions_onehot, dim=1, keepdim=True)

        prob = torch.clamp(prob, 1e-10, 1.0)
        old_prob = torch.clamp(old_prob, 1e-10, 1.0)

        ratio = prob / old_prob

        p1 = ratio * advantages
        p2 = torch.clamp(ratio, 1 - self.clip, 1 + self.clip) * advantages

        actor_loss = -torch.mean(torch.min(p1, p2))

        entropy = -torch.mean(new_probs * torch.log(new_probs + 1e-10))
        entropy *= self.entropy_coef

        return actor_loss - entropy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F     
GAMMA = 0.99
LAMBDA = 0.95  
MEAN_AVERAGE = 100
NB_EPISODES = 1000 if torch.cuda.is_available() else 50 

class PPOAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, clip=0.2, entropy_coef=0.001, 
                 gamma=GAMMA, lam=LAMBDA, ppo_epochs=12, batch_size=32, hidden_size=64):
        
        self.action_dim = action_dim
        self.clip = clip
        self.entropy_coef = entropy_coef
        self.ppo_epochs = ppo_epochs
        self.batch_size = batch_size
        self.gamma = gamma
        self.lam = lam
        self.lr=lr

        # Réseaux
        self.policy = PolicyNetwork(hidden_size, state_dim, action_dim).to(device)
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)
        self.value_net = ValueNetwork(hidden_size, state_dim).to(device)
        self.critic_optimizer = torch.optim.Adam(self.value_net.parameters(), lr=lr)

    def update_lr(self, new_lr):
        """ Met à jour le Learning Rate pour les optimizers de l'Actor et du Critic. """
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = new_lr
        for param_group in self.critic_optimizer.param_groups:
            param_group['lr'] = new_lr
        print(f"\n--- Learning Rate mis à jour à {new_lr:.1e} ---")    

    def select_action(self, state):
        """
        Sélectionne une action, retourne l'action (scalar), son log_prob (scalar) et la valeur (scalar).
        """
        if not isinstance(state, torch.Tensor):
            state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        
        with torch.no_grad():
            probs = self.policy(state)
            dist = Categorical(probs)
            
            action = dist.sample()
            log_prob = dist.log_prob(action)
            value = self.value_net(state).squeeze(-1) 

        return action, log_prob, value 

    def compute_gae(self, rewards, dones, values, next_value):
        """
        Calcule les Avantages Généralisés (GAE) et les Retours (Returns) pour la phase d'entraînement.
        """
        T = len(rewards)
        advantages = torch.zeros(T, device=device)
        gae = 0.0
        
        if not isinstance(next_value, torch.Tensor):
            next_value = torch.as_tensor(next_value, dtype=torch.float32, device=device)
        
        if not isinstance(rewards, torch.Tensor):
             rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device)
             dones = torch.as_tensor(dones, dtype=torch.float32, device=device)
             values = torch.as_tensor(values, dtype=torch.float32, device=device)

        for t in reversed(range(T)):
            mask = 1.0 - dones[t] 
            
            if t == T - 1:
                delta = rewards[t] + self.gamma * next_value * mask - values[t]
            else:
                delta = rewards[t] + self.gamma * values[t+1] * mask - values[t]
            
            gae = delta + self.gamma * self.lam * mask * gae
            advantages[t] = gae
            
        returns = advantages + values
        return advantages, returns
    
    def train(self, states, actions, old_log_probs, advantages, returns):
        """
        Effectue les mises à jour des réseaux Actor et Critic via mini-batchs.
        """
        states = torch.as_tensor(states, dtype=torch.float32, device=device)
        actions = torch.as_tensor(actions, dtype=torch.int64, device=device)
        old_log_probs = torch.as_tensor(old_log_probs, dtype=torch.float32, device=device).detach()
        advantages = torch.as_tensor(advantages, dtype=torch.float32, device=device).detach()
        returns = torch.as_tensor(returns, dtype=torch.float32, device=device).detach()

        T = len(states)
        total_loss = 0.0
        
      
        for _ in range(self.ppo_epochs):
            idx = torch.randperm(T, device=device) 
            
            for start in range(0, T, self.batch_size):
                end = start + self.batch_size
                b = idx[start:end]
                
                s_b = states[b]
                a_b = actions[b]
                oldlp_b = old_log_probs[b]
                adv_b = advantages[b]
                ret_b = returns[b]
                
                adv_b = (adv_b - adv_b.mean()) / (adv_b.std() + 1e-8)
                

                current_values = self.value_net(s_b).squeeze(-1)
                critic_loss = F.mse_loss(current_values, ret_b)

                self.critic_optimizer.zero_grad()
                critic_loss.backward()
                nn.utils.clip_grad_norm_(self.value_net.parameters(), 0.5)
                self.critic_optimizer.step()

             
                new_probs = self.policy(s_b)
                new_dist = Categorical(new_probs)
                log_probs = new_dist.log_prob(a_b)         
                
              
                ratio = torch.exp(log_probs - oldlp_b)    
                
                
                p1 = ratio * adv_b
                p2 = torch.clamp(ratio, 1.0 - self.clip, 1.0 + self.clip) * adv_b
                actor_loss = -torch.min(p1, p2).mean() 

               
                entropy = new_dist.entropy().mean()
                actor_loss = actor_loss - self.entropy_coef * entropy 

                self.optimizer.zero_grad()
                actor_loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optimizer.step()
                
                total_loss += (actor_loss.item() + critic_loss.item())

        n_updates = self.ppo_epochs * max(1, (T + self.batch_size - 1) // self.batch_size)
        return total_loss / n_updates


In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.probs = [] 

    def clear(self):
        self.__init__()

In [ ]:

GAMMA = 0.99
LAMBDA = 0.95  
MEAN_AVERAGE = 100
NB_EPISODES = 2000 if torch.cuda.is_available() else 50 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LR_START = 1e-4 
LR_END = 1e-5   
LR_SWITCH_EPISODE = 500 

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
MAX_ROLLOUT_STEPS = 1000
agent = PPOAgent(state_dim=state_dim, action_dim=action_dim, lr=LR_START) 
buffer = RolloutBuffer()

rewards = []
mean_rewards = []
losses = []
mean_losses = []

print(f"Démarrage de l'entraînement sur {device} pour {NB_EPISODES} épisodes...")

for i_episode in range(NB_EPISODES):
    
    current_episode_index = i_episode + 1
    
    # Diminuer le LR à 1e-5 à l'épisode 500 pour la stabilisation( car 1e-4 était trop rapide).
    if current_episode_index == LR_SWITCH_EPISODE:
        agent.update_lr(LR_END)

    rollout_steps = MAX_ROLLOUT_STEPS

    # Réinitialisation de l'environnement
    obs, _ = env.reset()
    state = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0) 
    reward_episode = 0
    
    for step in range(rollout_steps):
    
        action_tensor, log_prob_tensor, value_tensor = agent.select_action(state)
    
        action_to_env = action_tensor.item() 
        obs, reward, terminated, truncated, _ = env.step(action_to_env)
        done = terminated or truncated
    
        next_state = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    
        # ---- Stockage des données ----
        buffer.states.append(state.squeeze(0).cpu()) 
        buffer.actions.append(action_tensor.cpu())
        buffer.rewards.append(torch.tensor(reward, dtype=torch.float32))
        buffer.dones.append(torch.tensor(float(done), dtype=torch.float32))
        buffer.log_probs.append(log_prob_tensor.cpu())
        buffer.values.append(value_tensor.cpu())
    
        state = next_state 
        reward_episode += reward
    
        if done:
            obs, _ = env.reset()
            state = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

        if len(buffer.states) >= MAX_ROLLOUT_STEPS:
            break

    if len(buffer.states) == 0:
        buffer.clear()
        continue

    # Stacking et envoi sur le device
    states = torch.stack(buffer.states).to(device)                           
    actions = torch.stack(buffer.actions).to(device).squeeze(1)              
    old_log_probs = torch.stack(buffer.log_probs).to(device).squeeze(1)      
    values_old = torch.stack(buffer.values).to(device).squeeze()             
    rewards_tensor = torch.stack(buffer.rewards).to(device)                  
    dones_tensor = torch.stack(buffer.dones).to(device)                      

    
    with torch.no_grad():
        next_state_value = agent.value_net(state).squeeze()

    # Calcul GAE (Avantages et Returns)
    advantages, returns = agent.compute_gae(
        rewards_tensor, dones_tensor, values_old, next_state_value
    )
    
    # Mise à jour des réseaux
    loss_episode = agent.train(states, actions, old_log_probs, advantages.detach(), returns.detach())

    buffer.clear()

    # Phase 3 : Logging et Affichage
    rewards.append(reward_episode)
    losses.append(loss_episode)
    
    if len(losses) >= MEAN_AVERAGE:
        mean_losses.append(np.mean(losses[-MEAN_AVERAGE:]))

    if len(rewards) >= MEAN_AVERAGE:
        mean_rewards.append(np.mean(rewards[-MEAN_AVERAGE:]))
    
    if current_episode_index % 10 == 0:
        print(f"Épisode {current_episode_index}/{NB_EPISODES} | Récompense: {reward_episode:.2f} | Perte Moyenne: {loss_episode:.4f} | Moyenne 100 Ép: {mean_rewards[-1]:.2f} | LR Actuel: {agent.lr:.1e}" if len(mean_rewards) > 0 else f"Épisode {current_episode_index}/{NB_EPISODES} | Récompense: {reward_episode:.2f} | Perte Moyenne: {loss_episode:.4f} | LR Actuel: {agent.lr:.1e}")
        plot(rewards, mean_rewards, ylabel='Reward')
        if len(mean_losses) > 0: 
                plot(losses, mean_losses, ylabel='Perte', fig_num=2)

# Fin de l'entraînement
env.close()

# Affichage final
plot(rewards, mean_rewards, ylabel='Reward', show_result=True)
plot(losses, mean_losses, ylabel='Loss', show_result=True)

In [ ]:
# --- Sauvegarde des Modèles ---

print(f"\n--- Début de la Sauvegarde des Modèles ---")
#POLICY_SAVE_PATH= "LunarLander_PPO_G99_LR1e-4_CLIP20_AVG90_96_policy.pth"
#POLICY_SAVE_PATH= "LunarLander_PPO_G99_LR1e-4_500ep_LR1e-5_epochs15_1000steps_1300ep_CLIP20_AVG90_96_policy.pth"
POLICY_SAVE_PATH= "LunarLander_PPO_G99_LR1e-4_500ep_LR1e-5_epochs20_1000steps_2000ep_CLIP20_AVG90_96_policy.pth"
print(f"Sauvegarde de l'Actor (Policy Network) dans {POLICY_SAVE_PATH}...")
# On sauvegarde le dictionnaire d'état (les poids)
torch.save(agent.policy.state_dict(), POLICY_SAVE_PATH)

#VALUE_SAVE_PATH="LunarLander_PPO_G99_LR1e-4_CLIP20_AVG90_96_value.pth"
#VALUE_SAVE_PATH="LunarLander_PPO_G99_LR1e-4_LR1e-5_epochs15_1000steps_1300ep_CLIP20_AVG90_96_value.pth"
VALUE_SAVE_PATH="LunarLander_PPO_G99_LR1e-4_LR1e-5_epochs20_1000steps_2000ep_CLIP20_AVG90_96_value.pth"

print(f"Sauvegarde du Critic (Value Network) dans {VALUE_SAVE_PATH}...")
# On sauvegarde le dictionnaire d'état (les poids)
torch.save(agent.value_net.state_dict(), VALUE_SAVE_PATH)

print("Sauvegarde terminée. Les fichiers .pth contiennent les poids entraînés.")

In [ ]:
import torch
import gymnasium as gym
import numpy as np
from tqdm import tqdm 
import os
import imageio.v2 as imageio 
import matplotlib.pyplot as plt
from IPython import display 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Configuration de la vidéo ---
VIDEO_DIR = "videos_lunar_lander2" 
NB_TEST_EPISODES = 5 
FRAME_RATE = 30 

try:
  
    base_name = file_name_base.replace('_policy', '').replace('_value', '')
except NameError:
 
    base_name = "LunarLander_PPO_TEST"

FINAL_VIDEO_NAME = f"{base_name}_Video_{NB_TEST_EPISODES}Ep.mp4"
OUTPUT_PATH = os.path.join(VIDEO_DIR, FINAL_VIDEO_NAME) 


os.makedirs(VIDEO_DIR, exist_ok=True) 

all_frames = [] 


test_env = gym.make("LunarLander-v3", render_mode="rgb_array") 


for episode in tqdm(range(NB_TEST_EPISODES), desc=f"Collecting Frames for Condensed Video ({NB_TEST_EPISODES} Episodes)"):
    
    obs, _ = test_env.reset()
    
    state = torch.as_tensor(obs, dtype=torch.float32, device='cpu').unsqueeze(0) 
    done = False
    total_reward = 0
    step_count = 0
    
    # Affichage interactif pour le premier épisode seulement
    do_live_plot = (episode == 0)


    if episode > 0:
    
        start_frame = test_env.render()
        for _ in range(10): # 10 frames = 0.33 secondes à 30 FPS
             all_frames.append(start_frame)


    while not done:
        with torch.no_grad(): 

           action_tensor, _, _ = agent.select_action(state.to(device)) 
        

        action_to_env = action_tensor.item() 
        next_obs, reward, terminated, truncated, info = test_env.step(action_to_env)
        
        done = terminated or truncated
        total_reward += reward
        step_count += 1
        
      
        state = torch.as_tensor(next_obs, dtype=torch.float32, device='cpu').unsqueeze(0)

   
        frame = test_env.render()
        
     
        all_frames.append(frame) 
            
      
        if do_live_plot and 'display' in locals():
            plt.figure(figsize=(6, 6))
            plt.imshow(frame)
            plt.axis('off')
            plt.title(f"Episode: {episode + 1}/{NB_TEST_EPISODES} | Step: {step_count} | Reward: {total_reward:.2f}")
       
            display.clear_output(wait=True)
            display.display(plt.gcf())
            
    print(f"\nEpisode {episode + 1} terminé | Récompense totale: {total_reward:.2f} | Steps: {step_count}")


test_env.close()

if all_frames:
    try:

        imageio.mimsave(OUTPUT_PATH, all_frames, fps=FRAME_RATE, quality=8) 
        print(f"\n Compilation ({len(all_frames)} frames) enregistrée avec succès : {OUTPUT_PATH}")
    except Exception as e:
        print(f"\n Erreur lors de la compilation video: {e}")
else:
     print("\n Pas de frame.")

print(f"\nTerminé. Le fichier unique est disponible dans le dossier {VIDEO_DIR}.")